<a href="https://colab.research.google.com/github/denisshaf/colab/blob/main/Lab1_Profiling%2BTriton.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
from torch.profiler import profile, schedule, ProfilerActivity

assert torch.cuda.is_available(), "Нужна видеокарта NVIDIA (локально или в Google Colab)"
device = torch.device("cuda")
print(f"GPU: {torch.cuda.get_device_name(0)}")


# 1. Наивная реализация (имитация любого кастомного слоя в PyTorch без готового C++ ядра)
def naive_softmax(x: torch.Tensor) -> torch.Tensor:
    # Читает X, пишет max -> 1-й проход по HBM
    x_max = x.max(dim=-1, keepdim=True).values
    # Читает X и max, пишет z -> 2-й проход по HBM
    z = x - x_max
    # Читает z, пишет exp -> 3-й проход по HBM
    numerator = torch.exp(z)
    # Читает exp, пишет sum -> 4-й проход по HBM
    denominator = numerator.sum(dim=-1, keepdim=True)
    # Читает exp и sum, пишет результат -> 5-й проход по HBM
    return numerator / denominator


# Точный замер времени на GPU через CUDA Events (без накладных расходов CPU)
def benchmark_ms(fn, x: torch.Tensor, warmup: int = 25, rep: int = 100) -> float:
    for _ in range(warmup):
        fn(x)
    torch.cuda.synchronize()

    start_events = [torch.cuda.Event(enable_timing=True) for _ in range(rep)]
    end_events = [torch.cuda.Event(enable_timing=True) for _ in range(rep)]

    for i in range(rep):
        start_events[i].record()
        fn(x)
        end_events[i].record()

    torch.cuda.synchronize()
    times = [s.elapsed_time(e) for s, e in zip(start_events, end_events)]
    return sum(times) / len(times)


# Размерность, типичная для матрицы Attention (4096 токенов x 4096)
M, N = 4096, 4096
x = torch.randn((M, N), device=device, dtype=torch.float16)

# Проверка корректности
ref = torch.softmax(x, dim=-1)
out_naive = naive_softmax(x)
assert torch.allclose(ref, out_naive, atol=1e-3), "Расхождение в расчетах!"

# Замеры времени
ms_naive = benchmark_ms(naive_softmax, x)
ms_compiled = benchmark_ms(torch.compile(naive_softmax), x)
ms_pytorch = benchmark_ms(lambda t: torch.softmax(t, dim=-1), x)

# Расчет эффективной пропускной способности памяти (GB/s)
# Идеальный fused-алгоритм читает матрицу 1 раз и пишет 1 раз: 2 * M * N * 2 байта (FP16)
total_bytes = 2 * x.numel() * x.element_size()
gb_s = lambda ms: (total_bytes / (ms * 1e-3)) / 1e9

print("\n--- Результаты бенчмарка ---")
print(f"Naive PyTorch:    {ms_naive:.3f} ms | {gb_s(ms_naive):6.1f} GB/s")
print(f"torch.compile:    {ms_compiled:.3f} ms | {gb_s(ms_compiled):6.1f} GB/s")
print(f"PyTorch native:   {ms_pytorch:.3f} ms | {gb_s(ms_pytorch):6.1f} GB/s")

# 2. Вскрываем подкапотные CUDA-кернелы наивной версии через PyTorch Profiler
print("\n--- Профайлер: какие CUDA-ядра реально запускает Naive PyTorch ---")

# Прогрев: ядра уже загружены, кэши аллокатора заполнены
for _ in range(10):
    naive_softmax(x)
torch.cuda.synchronize()

# schedule: шаг 1 — пропуск, шаги 2–3 — прогрев профайлера (инициализация CUPTI
# не попадает в результат), шаг 4 — запись
with profile(
    activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA],
    profile_memory=True,
    record_shapes=True,
    schedule=schedule(wait=1, warmup=2, active=1, repeat=1),
) as prof:
    for _ in range(4):
        naive_softmax(x)
        torch.cuda.synchronize()
        prof.step()

print(prof.key_averages().table(sort_by="cuda_time_total", row_limit=15, max_name_column_width=200))

# Полные имена ядер (без обрезки), только собственное время GPU
print("\n--- Полные имена CUDA-ядер ---")
for e in sorted(prof.key_averages(), key=lambda e: e.self_device_time_total, reverse=True):
    if e.self_device_time_total > 0:
        print(f"{e.self_device_time_total:9.1f} us  x{e.count}  {e.key}")

# Таймлайн для https://ui.perfetto.dev
prof.export_chrome_trace("naive_trace.json")
print("\nТрассировка сохранена в naive_trace.json")

GPU: Tesla T4

--- Результаты бенчмарка ---
Naive PyTorch:    1.565 ms |   42.9 GB/s
torch.compile:    0.289 ms |  232.4 GB/s
PyTorch native:   0.537 ms |  125.0 GB/s

--- Профайлер: какие CUDA-ядра реально запускает Naive PyTorch ---
--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                                                                                                                                                                    Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg     Self CUDA   Self CUDA %    CUDA total  CUDA time avg       CPU Mem  Self CPU Mem      CUDA 

In [ ]:
import torch
import triton
import triton.language as tl

assert torch.cuda.is_available(), "Нужна видеокарта NVIDIA"

# -----------------------------------------------------------------------------
# ЗАДАЧА НА ЗАВТРА: НАПИСАТЬ ЭТО ЯДРО
# -----------------------------------------------------------------------------
@triton.jit
def fused_softmax_kernel(
    output_ptr,     # Указатель на матрицу результата в памяти (HBM)
    input_ptr,      # Указатель на исходную матрицу в памяти (HBM)
    input_row_stride, # Шаг в памяти для перехода на следующую строку
    output_row_stride,
    n_cols,         # Количество колонок (размер строки)
    BLOCK_SIZE: tl.constexpr # Размер блока данных, который заберем в SRAM
):
    # 1. Получите индекс текущей строки, которую обрабатывает этот программный блок
    pid = tl.program_id(axis=0)

    # 2. Вычислите указатель (адрес в памяти) на первый элемент этой строки в input_ptr
    row_ptr = input_ptr + pid * input_row_stride

    # 3. Сгенерируйте вектор смещений от 0 до BLOCK_SIZE - 1
    offs = tl.arange(0, BLOCK_SIZE)

    # 4. Получите вектор указателей на все элементы текущей строки
    ptrs = row_ptr + offs

    # 5. Создайте булеву маску, отсекающую элементы за пределами n_cols
    mask = offs < n_cols

    # 6. Загрузите строку из HBM в SRAM. Элементы вне маски должны стать -inf
    x = tl.load(ptrs, mask=mask, other=float("-inf"))

    # 7. Посчитайте Softmax в регистрах/SRAM: exp(x - max(x)) / sum(exp(x - max(x)))
    # Важно: для стабильности вычислений лучше перевести загруженные fp16 данные в tl.float32,
    # а перед сохранением вернуть обратно в исходный тип (row.to(output_ptr.dtype.element_ty))
    x = x.to(tl.float32)
    x_max = tl.max(x)
    out = tl.exp(x - x_max) / tl.sum(tl.exp(x - x_max))

    # 8. Вычислите указатели для записи в output_ptr и сохраните результат с той же маской
    out_row_ptr = output_ptr + pid * output_row_stride
    output_ptrs = out_row_ptr + offs
    tl.store(output_ptrs, out, mask=mask)

# -----------------------------------------------------------------------------
# ОБВЯЗКА (Это трогать не нужно)
# -----------------------------------------------------------------------------
def triton_softmax(x: torch.Tensor) -> torch.Tensor:
    n_rows, n_cols = x.shape
    # Размер блока должен быть степенью двойки. Возьмем ближайшую сверху.
    BLOCK_SIZE = triton.next_power_of_2(n_cols)

    # Выделяем пустую память под результат
    y = torch.empty_like(x)

    # Запускаем ядро: одна программа (грид) на каждую строку матрицы
    grid = (n_rows, )
    fused_softmax_kernel[grid](
        y, x,
        x.stride(0), y.stride(0),
        n_cols,
        num_warps=4, # Подсказка компилятору
        BLOCK_SIZE=BLOCK_SIZE,
    )
    return y

# Замеры времени и пропускной способности (как мы делали в предыдущем скрипте)
def benchmark_ms(fn, x, warmup=25, rep=100):
    for _ in range(warmup): fn(x)
    torch.cuda.synchronize()
    start = [torch.cuda.Event(enable_timing=True) for _ in range(rep)]
    end = [torch.cuda.Event(enable_timing=True) for _ in range(rep)]
    for i in range(rep):
        start[i].record()
        fn(x)
        end[i].record()
    torch.cuda.synchronize()
    return sum(s.elapsed_time(e) for s, e in zip(start, end)) / rep

if __name__ == "__main__":
    M, N = 4096, 4096
    x = torch.randn((M, N), device='cuda', dtype=torch.float16)

    # 1. Проверяем математику
    ref = torch.softmax(x, dim=-1)
    # Раскомментируйте завтра, когда напишете ядро
    out = triton_softmax(x)
    assert torch.allclose(ref, out, atol=1e-3), "Математика не сошлась!"
    print("Успех: математика Triton-ядра совпадает с PyTorch!")

    # 2. Замеряем скорость
    ms_triton = benchmark_ms(triton_softmax, x)
    ms_torch = benchmark_ms(lambda t: torch.softmax(t, dim=-1), x)

    gb_s = lambda ms: (2 * x.numel() * x.element_size() / (ms * 1e-3)) / 1e9

    print("\n--- Пропускная способность (GB/s) ---")
    print(f"PyTorch C++ Backend: {gb_s(ms_torch):.1f} GB/s")
    print(f"Ваш Triton Kernel:   {gb_s(ms_triton):.1f} GB/s")

Успех: математика Triton-ядра совпадает с PyTorch!

--- Пропускная способность (GB/s) ---
PyTorch C++ Backend: 74.2 GB/s
Ваш Triton Kernel:   234.9 GB/s


In [ ]:
from torch.profiler import profile, record_function, schedule, ProfilerActivity

# Компилируем заранее, чтобы в профайлер не попал оверхед первой компиляции
compiled_softmax = torch.compile(naive_softmax)

# Прогрев всех четырех вариантов
for _ in range(10):
    naive_softmax(x)
    compiled_softmax(x)
    torch.softmax(x, dim=-1)
    triton_softmax(x)
torch.cuda.synchronize()

with profile(
    activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA],
    profile_memory=True,
    record_shapes=True,
    schedule=schedule(wait=1, warmup=2, active=1, repeat=1),
) as prof:
    for _ in range(4):
        with record_function("=== 1. NAIVE ==="):
            naive_softmax(x)
        with record_function("=== 2. TORCH_COMPILE ==="):
            compiled_softmax(x)
        with record_function("=== 3. PYTORCH_NATIVE ==="):
            torch.softmax(x, dim=-1)
        with record_function("=== 4. MY_TRITON ==="):
            triton_softmax(x)
        torch.cuda.synchronize()
        prof.step()

print(prof.key_averages().table(sort_by="cuda_time_total", row_limit=25, max_name_column_width=90))
prof.export_chrome_trace("comparison_trace.json")

------------------------------------------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                                                      Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg     Self CUDA   Self CUDA %    CUDA total  CUDA time avg       CPU Mem  Self CPU Mem      CUDA Mem  Self CUDA Mem    # of Calls  
------------------------------------------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                                             ProfilerStep*         3.33%     120.129us        99.93